<a href="https://colab.research.google.com/github/yosie111/cadcam/blob/main/stage2_cadcam_ch4_heb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# שלב 2 — פתרון OO סטנדרטי: תת-מחלקות מיוחדות (פרק 4)

**מקור:** *Design Patterns Explained* (Shalloway & Trott), פרק 4 — "A Standard Object-Oriented Solution"

---

## הרעיון של פרק 4 במילים שלי

בפרק 3 ראינו את הכאב של V1 הפרוצדורלי. עכשיו ננסה פתרון **מונחה-עצמים סטנדרטי**.

הרעיון פשוט: אם אני יודע לפתור את הבעיה ל-**Slot**, אני יכול לחזור על אותו פתרון ל-Hole, Cutout, Special, Irregular.

### Figure 4-1: התחלה עם Slot
יוצרים מחלקה אבסטרקטית `SlotFeature` עם שתי נגזרות:
- `V1Slot` — שומר handle + feature_id, קורא לפונקציות V1
- `V2Slot` — שומר הפניה ל-OOGFeature, מאציל אליו

### Figure 4-2: מרחיבים לכל סוגי ה-features
אותו דפוס חוזר ל-Hole, Cutout, Special, Irregular.
תוצאה: **5 סוגי ביניים × 2 גרסאות = 10 מחלקות עלים** + 6 מחלקות בסיס.

### Figure 4-3: החיבור למערכות בפועל
- V1xxx שומרים handle + ID ופונים ל-V1 System
- V2xxx שומרים הפניה ל-OOGxxx ומאצילים אליו
- מערכת המומחה מקבלת `Feature` ולא יודעת מאיזו גרסה הוא בא

### מה שפרק 4 אומר במפורש
> *"It is not a great solution, but it is a solution that would work."*

הפתרון הזה **עובד** — אבל הוא רק צעד ביניים. הבעיות שלו ייחשפו בדמו.

---

## תרשים מבנה (Figure 4-2 + Figure 4-3)

```
           Model ◇──────────── Feature (ABC)
                                  △  feature_type()
                                  │  get_x(), get_y()
                ┌─────────┬───────┼──────────┬──────────┐
          SlotFeature  HoleFeature CutoutFeature SpecialFeature IrregularFeature
          +get_width() +get_radius() +get_width() +get_special_code() +get_bbox_width()
          +get_height()+get_depth()  +get_height()+...             +get_bbox_height()
          +get_angle()              +get_angle()
          +get_depth()
              △                △           △            △              △
          ┌───┴───┐       ┌───┴───┐   ┌───┴───┐   ┌───┴───┐    ┌────┴────┐
        V1Slot  V2Slot  V1Hole V2Hole V1Cut V2Cut V1Spc V2Spc V1Irr  V2Irr
          │       │       │      │     │      │     │      │     │       │
          │       ▼       │      ▼     │      ▼     │      ▼     │       ▼
          │    OOGSlot    │   OOGHole  │   OOGCut   │  OOGSpc    │   OOGIrr
          │               │            │            │             │
          └───────────────┴────────────┴────────────┴─────────────┘
                                  ▼
                              V1 System
```

### סכימת הספירה

| גרסה     | מחלקות עלים | סה"כ מחלקות |
|:---------|:------------|:-----------|
| V1 בלבד  | 5           | 11         |
| V1 + V2  | 10          | **16**     |
| V1+V2+V3 | 15          | **21**     |
| נוסחה     | 5 × N       | 6 + 5N    |

← **פיצוץ מחלקות (Class Explosion)!**

---
## חלק A: ספריית V1 הפרוצדורלית + מסד נתונים
זהה לשלב 1 — אנחנו **לא יכולים לשנות** את V1. זו ספריית C חיצונית.

In [ ]:
# ════════════════════════════════════════════════════════════
# חלק A: ספריית V1 הפרוצדורלית + מסד נתונים (מועתק משלב 1)
# ════════════════════════════════════════════════════════════

from __future__ import annotations
from abc import ABC, abstractmethod

# קבועים לשדות ה-tuple
FID=0; FTYPE=1; X=2; Y=3; WIDTH=4; HEIGHT=5; ANGLE=6; DEPTH=7; RADIUS=8; SPECIAL_CODE=9

_V1_DATABASE: dict[str, dict] = {
    "PART-001": {
        "sheet_width": 200.0, "sheet_height": 150.0, "material": "Aluminum-6061",
        "features": [
            ( 1, "slot",     10.0,  30.0,  40.0,  8.0,  0.0, 2.0, None,  None),
            ( 2, "slot",     10.0,  60.0,  40.0,  8.0,  0.0, 2.0, None,  None),
            ( 3, "hole",     30.0,  60.0,  12.0, 12.0,  0.0, 2.0,  6.0,  None),
            ( 4, "hole",    120.0,  75.0,  30.0, 30.0,  0.0, 2.0, 15.0,  None),
            ( 5, "cutout",   80.0,  40.0,  35.0, 25.0,  0.0, 2.0, None,  None),
            ( 6, "cutout",  130.0,  90.0,  25.0, 25.0, 45.0, 2.0, None,  None),
            ( 7, "special",  90.0,  30.0,  20.0, 20.0,  0.0, 2.0, None,  "ELEC-OUTLET"),
            ( 8, "irregular",140.0, 110.0,  25.0, 15.0, 30.0, 2.0, None,  None),
        ],
    },
}

_open_handles: dict[int, str] = {}
_next_handle: int = 0

def v1_open_model(name: str) -> int:
    global _next_handle
    if name not in _V1_DATABASE: raise ValueError(f"V1: {name!r} לא נמצא")
    _next_handle += 1; _open_handles[_next_handle] = name; return _next_handle

def v1_close_model(h: int) -> None: _open_handles.pop(h, None)
def _v1m(h: int) -> dict: return _V1_DATABASE[_open_handles[h]]
def _v1f(h: int, fid: int) -> tuple:
    for r in _v1m(h)["features"]:
        if r[FID] == fid: return r
    raise ValueError(f"V1: feature {fid} לא נמצא")

def v1_get_num_features(h): return len(_v1m(h)["features"])
def v1_get_feature_id(h, i): return _v1m(h)["features"][i][FID]
def v1_get_feature_type(h, i): return _v1m(h)["features"][i][FTYPE]

# Getters ייעודיים לכל סוג (22 פונקציות — הכאב של V1)
def v1_get_x_for_slot(h,fid): return _v1f(h,fid)[X]
def v1_get_y_for_slot(h,fid): return _v1f(h,fid)[Y]
def v1_get_w_for_slot(h,fid): return _v1f(h,fid)[WIDTH]
def v1_get_h_for_slot(h,fid): return _v1f(h,fid)[HEIGHT]
def v1_get_angle_for_slot(h,fid): return _v1f(h,fid)[ANGLE]
def v1_get_depth_for_slot(h,fid): return _v1f(h,fid)[DEPTH]

def v1_get_x_for_hole(h,fid): return _v1f(h,fid)[X]
def v1_get_y_for_hole(h,fid): return _v1f(h,fid)[Y]
def v1_get_radius_for_hole(h,fid): return _v1f(h,fid)[RADIUS]
def v1_get_depth_for_hole(h,fid): return _v1f(h,fid)[DEPTH]

def v1_get_x_for_cutout(h,fid): return _v1f(h,fid)[X]
def v1_get_y_for_cutout(h,fid): return _v1f(h,fid)[Y]
def v1_get_w_for_cutout(h,fid): return _v1f(h,fid)[WIDTH]
def v1_get_h_for_cutout(h,fid): return _v1f(h,fid)[HEIGHT]
def v1_get_angle_for_cutout(h,fid): return _v1f(h,fid)[ANGLE]

def v1_get_x_for_special(h,fid): return _v1f(h,fid)[X]
def v1_get_y_for_special(h,fid): return _v1f(h,fid)[Y]
def v1_get_code_for_special(h,fid): return _v1f(h,fid)[SPECIAL_CODE]

def v1_get_x_for_irregular(h,fid): return _v1f(h,fid)[X]
def v1_get_y_for_irregular(h,fid): return _v1f(h,fid)[Y]
def v1_get_bbox_w_for_irregular(h,fid): return _v1f(h,fid)[WIDTH]
def v1_get_bbox_h_for_irregular(h,fid): return _v1f(h,fid)[HEIGHT]

print("ספריית V1 מוכנה (מועתקת משלב 1)")

---
## חלק B: מערכת V2 — OOG (Object Oriented Geometry)
V2 מחזיר **אובייקטים** — הרבה יותר נחמד מ-V1.  
אבל הממשק שלהם **שונה** ממה שאנחנו צריכים:

| מה אנחנו רוצים | מה V2 נותן | הבעיה |
|:---------------|:-----------|:------|
| `get_width()`  | `get_length()` | שם שונה! |
| `get_height()` | `get_width()`  | שם שונה! |
| —              | `get_edge_type()` | מושג חדש שלא קיים ב-V1 |

הספר: *"These objects do NOT have the correct interface because I did not design them."*

In [ ]:
# ════════════════════════════════════════════════════════════
# חלק B: מערכת V2 — OOG (Object Oriented Geometry)
# ════════════════════════════════════════════════════════════

class OOGFeature:
    """אובייקט Feature של V2 — ממשק OO נחמד אבל עם שמות שונים."""
    def __init__(self, ftype: str, x: float, y: float,
                 length: float, width: float, angle: float = 0.0,
                 depth: float = 1.0, radius: float | None = None,
                 special_code: str | None = None, edge_type: str = "square"):
        self._type = ftype; self._x = x; self._y = y
        self._length = length; self._width = width
        self._angle = angle; self._depth = depth
        self._radius = radius; self._special_code = special_code
        self._edge_type = edge_type

    def my_type(self) -> str: return self._type
    def get_x(self) -> float: return self._x
    def get_y(self) -> float: return self._y
    def get_length(self) -> float: return self._length      # != get_width שלנו!
    def get_width(self) -> float: return self._width        # != get_height שלנו!
    def get_angle(self) -> float: return self._angle
    def get_depth(self) -> float: return self._depth
    def get_radius(self) -> float: return self._radius or self._length / 2
    def get_special_code(self) -> str: return self._special_code or "?"
    def get_edge_type(self) -> str: return self._edge_type   # מושג חדש!
    def __repr__(self) -> str: return f"OOGFeature({self._type}@{self._x},{self._y})"


class OOGModel:
    """מודל של V2 — מכיל אוסף של OOGFeature."""
    def __init__(self, name: str, sw: float, sh: float,
                 mat: str, features: list[OOGFeature]):
        self._name = name; self._sw = sw; self._sh = sh
        self._mat = mat; self._features = features

    def get_model_name(self) -> str: return self._name
    def get_sheet_width(self) -> float: return self._sw
    def get_sheet_height(self) -> float: return self._sh
    def get_material(self) -> str: return self._mat
    def get_elements(self) -> list[OOGFeature]: return list(self._features)


_V2_DATABASE: dict[str, OOGModel] = {
    "PART-001": OOGModel("PART-001", 200, 150, "Aluminum-6061", [
        OOGFeature("slot",     10.0,  30.0, 40.0,  8.0,  0.0, 2.0, edge_type="rounded"),
        OOGFeature("slot",     10.0,  60.0, 40.0,  8.0,  0.0, 2.0, edge_type="rounded"),
        OOGFeature("hole",     30.0,  60.0, 12.0, 12.0,  0.0, 2.0, radius=6.0),
        OOGFeature("hole",    120.0,  75.0, 30.0, 30.0,  0.0, 2.0, radius=15.0),
        OOGFeature("cutout",   80.0,  40.0, 35.0, 25.0,  0.0, 2.0, edge_type="squared"),
        OOGFeature("cutout",  130.0,  90.0, 25.0, 25.0, 45.0, 2.0, edge_type="rounded"),
        OOGFeature("special",  90.0,  30.0, 20.0, 20.0,  0.0, 2.0, special_code="ELEC-OUTLET"),
        OOGFeature("irregular",140.0, 110.0, 25.0, 15.0, 30.0, 2.0),
    ]),
}

def v2_open_model(name: str) -> OOGModel:
    if name not in _V2_DATABASE: raise ValueError(f"V2: {name!r} לא נמצא")
    return _V2_DATABASE[name]

print("מערכת V2 (OOG) מוכנה")

---
## חלק C: היררכיית המחלקות של פרק 4 (Figure 4-1 → 4-2 → 4-3)
זהו **לב הפתרון של פרק 4**: מחלקת בסיס `Feature`, מחלקות ביניים לפי סוג, ומחלקות עלים לפי גרסה.

כל V1xxx **שומר handle + feature_id** ופונה לפונקציות V1.  
כל V2xxx **שומר הפניה ל-OOGFeature** ומאציל אליו (עם מיפוי שמות).

In [ ]:
# ════════════════════════════════════════════════════════════
# חלק C: היררכיית המחלקות של פרק 4
# ════════════════════════════════════════════════════════════

# ── Feature: מחלקת בסיס (Figure 4-3 — הממשק שמערכת המומחה מצפה לו) ──

class Feature(ABC):
    """מחלקת בסיס אבסטרקטית — הממשק שמערכת המומחה מצפה לו."""

    @abstractmethod
    def feature_type(self) -> str: ...

    @abstractmethod
    def get_x(self) -> float: ...

    @abstractmethod
    def get_y(self) -> float: ...


# ── SlotFeature → V1Slot / V2Slot (Figure 4-1) ──────────

class SlotFeature(Feature):
    """מחלקת ביניים ל-slot — מוסיפה מתודות ייחודיות."""
    def feature_type(self) -> str: return "slot"

    @abstractmethod
    def get_width(self) -> float: ...
    @abstractmethod
    def get_height(self) -> float: ...
    @abstractmethod
    def get_angle(self) -> float: ...
    @abstractmethod
    def get_depth(self) -> float: ...


class V1Slot(SlotFeature):
    """V1Slot: שומר handle + fid, קורא לפונקציות V1."""
    def __init__(self, handle: int, feature_id: int):
        self._handle = handle; self._fid = feature_id

    def get_x(self) -> float:      return v1_get_x_for_slot(self._handle, self._fid)
    def get_y(self) -> float:      return v1_get_y_for_slot(self._handle, self._fid)
    def get_width(self) -> float:  return v1_get_w_for_slot(self._handle, self._fid)
    def get_height(self) -> float: return v1_get_h_for_slot(self._handle, self._fid)
    def get_angle(self) -> float:  return v1_get_angle_for_slot(self._handle, self._fid)
    def get_depth(self) -> float:  return v1_get_depth_for_slot(self._handle, self._fid)


class V2Slot(SlotFeature):
    """V2Slot: שומר הפניה ל-OOGFeature, מאציל עם מיפוי שמות."""
    def __init__(self, oog: OOGFeature):
        self._oog = oog

    def get_x(self) -> float:      return self._oog.get_x()
    def get_y(self) -> float:      return self._oog.get_y()
    def get_width(self) -> float:  return self._oog.get_length()    # ← שם שונה!
    def get_height(self) -> float: return self._oog.get_width()     # ← שם שונה!
    def get_angle(self) -> float:  return self._oog.get_angle()
    def get_depth(self) -> float:  return self._oog.get_depth()


# ── HoleFeature → V1Hole / V2Hole ───────────────────────

class HoleFeature(Feature):
    """מחלקת ביניים ל-hole."""
    def feature_type(self) -> str: return "hole"

    @abstractmethod
    def get_radius(self) -> float: ...
    @abstractmethod
    def get_depth(self) -> float: ...


class V1Hole(HoleFeature):
    def __init__(self, handle: int, feature_id: int):
        self._handle = handle; self._fid = feature_id

    def get_x(self) -> float:      return v1_get_x_for_hole(self._handle, self._fid)
    def get_y(self) -> float:      return v1_get_y_for_hole(self._handle, self._fid)
    def get_radius(self) -> float: return v1_get_radius_for_hole(self._handle, self._fid)
    def get_depth(self) -> float:  return v1_get_depth_for_hole(self._handle, self._fid)


class V2Hole(HoleFeature):
    def __init__(self, oog: OOGFeature):
        self._oog = oog

    def get_x(self) -> float:      return self._oog.get_x()
    def get_y(self) -> float:      return self._oog.get_y()
    def get_radius(self) -> float: return self._oog.get_radius()
    def get_depth(self) -> float:  return self._oog.get_depth()


# ── CutoutFeature → V1Cutout / V2Cutout ─────────────────

class CutoutFeature(Feature):
    """מחלקת ביניים ל-cutout."""
    def feature_type(self) -> str: return "cutout"

    @abstractmethod
    def get_width(self) -> float: ...
    @abstractmethod
    def get_height(self) -> float: ...
    @abstractmethod
    def get_angle(self) -> float: ...


class V1Cutout(CutoutFeature):
    def __init__(self, handle: int, feature_id: int):
        self._handle = handle; self._fid = feature_id

    def get_x(self) -> float:      return v1_get_x_for_cutout(self._handle, self._fid)
    def get_y(self) -> float:      return v1_get_y_for_cutout(self._handle, self._fid)
    def get_width(self) -> float:  return v1_get_w_for_cutout(self._handle, self._fid)
    def get_height(self) -> float: return v1_get_h_for_cutout(self._handle, self._fid)
    def get_angle(self) -> float:  return v1_get_angle_for_cutout(self._handle, self._fid)


class V2Cutout(CutoutFeature):
    def __init__(self, oog: OOGFeature):
        self._oog = oog

    def get_x(self) -> float:      return self._oog.get_x()
    def get_y(self) -> float:      return self._oog.get_y()
    def get_width(self) -> float:  return self._oog.get_length()    # ← שם שונה!
    def get_height(self) -> float: return self._oog.get_width()     # ← שם שונה!
    def get_angle(self) -> float:  return self._oog.get_angle()


# ── SpecialFeature → V1Special / V2Special ───────────────

class SpecialFeature(Feature):
    """מחלקת ביניים ל-special."""
    def feature_type(self) -> str: return "special"

    @abstractmethod
    def get_special_code(self) -> str: ...


class V1Special(SpecialFeature):
    def __init__(self, handle: int, feature_id: int):
        self._handle = handle; self._fid = feature_id

    def get_x(self) -> float:           return v1_get_x_for_special(self._handle, self._fid)
    def get_y(self) -> float:           return v1_get_y_for_special(self._handle, self._fid)
    def get_special_code(self) -> str:  return v1_get_code_for_special(self._handle, self._fid)


class V2Special(SpecialFeature):
    def __init__(self, oog: OOGFeature):
        self._oog = oog

    def get_x(self) -> float:           return self._oog.get_x()
    def get_y(self) -> float:           return self._oog.get_y()
    def get_special_code(self) -> str:  return self._oog.get_special_code()


# ── IrregularFeature → V1Irregular / V2Irregular ────────

class IrregularFeature(Feature):
    """מחלקת ביניים ל-irregular."""
    def feature_type(self) -> str: return "irregular"

    @abstractmethod
    def get_bbox_width(self) -> float: ...
    @abstractmethod
    def get_bbox_height(self) -> float: ...


class V1Irregular(IrregularFeature):
    def __init__(self, handle: int, feature_id: int):
        self._handle = handle; self._fid = feature_id

    def get_x(self) -> float:           return v1_get_x_for_irregular(self._handle, self._fid)
    def get_y(self) -> float:           return v1_get_y_for_irregular(self._handle, self._fid)
    def get_bbox_width(self) -> float:  return v1_get_bbox_w_for_irregular(self._handle, self._fid)
    def get_bbox_height(self) -> float: return v1_get_bbox_h_for_irregular(self._handle, self._fid)


class V2Irregular(IrregularFeature):
    def __init__(self, oog: OOGFeature):
        self._oog = oog

    def get_x(self) -> float:           return self._oog.get_x()
    def get_y(self) -> float:           return self._oog.get_y()
    def get_bbox_width(self) -> float:  return self._oog.get_length()  # ← שם שונה!
    def get_bbox_height(self) -> float: return self._oog.get_width()   # ← שם שונה!


print("היררכיית פרק 4 מוכנה: 16 מחלקות (1 בסיס + 5 ביניים + 5×V1 + 5×V2)")

---
## חלק D: טעינת מודלים — יצירת האובייקטים הנכונים
הטוען בודק את סוג כל feature ויוצר את המחלקה הנכונה.  
לכל גרסה טבלת ניתוב משלה.

In [ ]:
# ════════════════════════════════════════════════════════════
# חלק D: טעינת מודלים
# ════════════════════════════════════════════════════════════

# טבלאות ניתוב — מיפוי מחרוזת סוג ← מחלקה
V1_CLASS_MAP: dict[str, type] = {
    "slot": V1Slot, "hole": V1Hole, "cutout": V1Cutout,
    "special": V1Special, "irregular": V1Irregular,
}

V2_CLASS_MAP: dict[str, type] = {
    "slot": V2Slot, "hole": V2Hole, "cutout": V2Cutout,
    "special": V2Special, "irregular": V2Irregular,
}


def load_v1_model(model_name: str) -> tuple[int, list[Feature]]:
    """
    פותחת מודל V1, יוצרת אובייקט Feature עבור כל feature.
    מחזירה (handle, רשימת_features).
    הערה: ה-handle חייב להישאר פתוח כל עוד ה-features בשימוש!
    """
    handle = v1_open_model(model_name)
    features: list[Feature] = []

    for i in range(v1_get_num_features(handle)):
        fid = v1_get_feature_id(handle, i)
        ftype = v1_get_feature_type(handle, i)

        cls = V1_CLASS_MAP.get(ftype)
        if cls is None:
            raise ValueError(f"סוג feature לא מוכר: {ftype!r}")
        features.append(cls(handle, fid))

    return handle, features


def load_v2_model(model_name: str) -> list[Feature]:
    """
    פותחת מודל V2, יוצרת אובייקט Feature עבור כל OOGFeature.
    מחזירה רשימת features.
    """
    oog_model = v2_open_model(model_name)
    features: list[Feature] = []

    for oog in oog_model.get_elements():
        ftype = oog.my_type()

        cls = V2_CLASS_MAP.get(ftype)
        if cls is None:
            raise ValueError(f"סוג OOG feature לא מוכר: {ftype!r}")
        features.append(cls(oog))

    return features


print("טוענים מוכנים: load_v1_model(), load_v2_model()")

---
## חלק E: מערכת המומחה — עובדת עם הממשק המשותף
מערכת המומחה מקבלת `list[Feature]` ומייצרת פקודות NC.  
**שימו לב:** היא עדיין צריכה `isinstance` כי לכל סוג יש מתודות שונות.  
זה חלק מהכאב של פרק 4 — הפולימורפיזם לא מלא.

In [ ]:
# ════════════════════════════════════════════════════════════
# חלק E: מערכת המומחה
# ════════════════════════════════════════════════════════════

def expert_system_process(features: list[Feature]) -> list[str]:
    """מערכת המומחה — מקבלת Feature ומייצרת פקודות NC."""
    commands: list[str] = []

    for feat in features:
        x, y = feat.get_x(), feat.get_y()

        # עדיין צריכים isinstance — הפולימורפיזם לא מלא!
        if isinstance(feat, SlotFeature):
            commands.append(
                f"NC: CUT_SLOT at ({x},{y}) "
                f"w={feat.get_width()} h={feat.get_height()} "
                f"angle={feat.get_angle()} depth={feat.get_depth()}"
            )
        elif isinstance(feat, HoleFeature):
            commands.append(
                f"NC: DRILL_HOLE at ({x},{y}) "
                f"r={feat.get_radius()} depth={feat.get_depth()}"
            )
        elif isinstance(feat, CutoutFeature):
            commands.append(
                f"NC: PUNCH_CUTOUT at ({x},{y}) "
                f"w={feat.get_width()} h={feat.get_height()} "
                f"angle={feat.get_angle()}"
            )
        elif isinstance(feat, SpecialFeature):
            commands.append(
                f"NC: PUNCH_SPECIAL at ({x},{y}) "
                f"code={feat.get_special_code()}"
            )
        elif isinstance(feat, IrregularFeature):
            commands.append(
                f"NC: MULTI_CUT at ({x},{y}) "
                f"bbox={feat.get_bbox_width()}x{feat.get_bbox_height()}"
            )

    return commands


print("מערכת מומחה מוכנה: expert_system_process()")

---
## דמו 1: טעינה מ-V1
טוענים את PART-001 דרך V1 ומעבירים למערכת המומחה.

In [ ]:
print("═" * 60)
print("  דמו 1: טעינת PART-001 דרך V1")
print("═" * 60)

v1_handle, v1_features = load_v1_model("PART-001")

print(f"\nנטענו {len(v1_features)} features מ-V1:\n")
for f in v1_features:
    print(f"  {f.__class__.__name__:<14} type={f.feature_type():<10} at ({f.get_x()},{f.get_y()})")

print(f"\nפקודות NC מ-V1:")
v1_commands = expert_system_process(v1_features)
for cmd in v1_commands:
    print(f"  {cmd}")

---
## דמו 2: טעינה מ-V2
אותו PART-001, אותה מערכת מומחה — אבל הפעם דרך V2.

In [ ]:
print("═" * 60)
print("  דמו 2: טעינת PART-001 דרך V2")
print("═" * 60)

v2_features = load_v2_model("PART-001")

print(f"\nנטענו {len(v2_features)} features מ-V2:\n")
for f in v2_features:
    print(f"  {f.__class__.__name__:<14} type={f.feature_type():<10} at ({f.get_x()},{f.get_y()})")

print(f"\nפקודות NC מ-V2:")
v2_commands = expert_system_process(v2_features)
for cmd in v2_commands:
    print(f"  {cmd}")

---
## דמו 3: ספירת מחלקות — בעיית הפיצוץ (Class Explosion)

In [ ]:
print("═" * 60)
print("  דמו 3: ספירת מחלקות")
print("═" * 60)

all_classes = [
    Feature,
    SlotFeature, V1Slot, V2Slot,
    HoleFeature, V1Hole, V2Hole,
    CutoutFeature, V1Cutout, V2Cutout,
    SpecialFeature, V1Special, V2Special,
    IrregularFeature, V1Irregular, V2Irregular,
]

print(f"\n  מחלקות בסיס אבסטרקטיות:  1  (Feature)")
print(f"  מחלקות ביניים לפי סוג:   5  (SlotFeature, HoleFeature, ...)")
print(f"  מחלקות V1 קונקרטיות:     5  (V1Slot, V1Hole, ...)")
print(f"  מחלקות V2 קונקרטיות:     5  (V2Slot, V2Hole, ...)")
print(f"  {'─'*35}")
print(f"  סה\"כ:                    {len(all_classes)} מחלקות")
print()
print(f"  כשיגיע V3:               +5 = 21 מחלקות")
print(f"  כשיגיע V4:               +5 = 26 מחלקות")
print(f"  נוסחה: 1 + 5 + (5 × N) ← פיצוץ מחלקות!")

---
## בדיקות

In [ ]:
print("─" * 60)
print("  בדיקות...")
print("─" * 60)

# כמות נכונה
assert len(v1_features) == 8
assert len(v2_features) == 8

# סוגים נכונים
assert isinstance(v1_features[0], SlotFeature) and isinstance(v1_features[0], V1Slot)
assert isinstance(v2_features[0], SlotFeature) and isinstance(v2_features[0], V2Slot)

# פולימורפיזם — שתיהן Feature
assert isinstance(v1_features[0], Feature)
assert isinstance(v2_features[0], Feature)

# נתונים זהים
assert v1_features[0].get_x() == v2_features[0].get_x() == 10.0
assert v1_features[0].get_width() == v2_features[0].get_width() == 40.0

# Hole
assert isinstance(v1_features[2], HoleFeature)
assert v1_features[2].get_radius() == v2_features[2].get_radius() == 6.0

# Special
assert isinstance(v1_features[6], SpecialFeature)
assert v1_features[6].get_special_code() == v2_features[6].get_special_code() == "ELEC-OUTLET"

# פקודות NC זהות!
assert v1_commands == v2_commands, "פקודות NC צריכות להיות זהות!"

# ספירת מחלקות
assert len(all_classes) == 16

# Handle שוחרר
v1_close_model(v1_handle)
assert len(_open_handles) == 0

print("\n  כל הבדיקות עברו בהצלחה.")

---

## מה עובד בפתרון הזה

1. **פולימורפיזם בסיסי** — מערכת המומחה מקבלת `list[Feature]` ולא צריכה לדעת מאיזו גרסה הנתונים באו.
2. **אנקפסולציה** — V1Slot שומר handle+fid בתוכו; V2Slot שומר OOGFeature. הצרכן לא רואה את החלקים הפנימיים.
3. **הסתרת מיפוי השמות** — V2Slot ממפה `get_length()` → `get_width()` פנימית. הצרכן לא יודע שהשמות שונים.
4. **מערכת מומחה אחת** — `expert_system_process` עובדת זהה על V1 ועל V2.

---

## מה כואב — ולמה זה לא הפתרון הסופי

### 1. פיצוץ מחלקות (Class Explosion)
16 מחלקות עבור 2 גרסאות. כשיגיע V3 — עוד 5. נוסחה: `6 + 5N`. הספר אומר:  
*"When V3 arrives: instead of 10 classes, we will have 15. This is certainly not a system I will have fun maintaining!"*

### 2. שכפול קוד
כל V2xxx חוזר על אותו דפוס: שמור OOGFeature, האצל אליו. `get_x()` ו-`get_y()` מועתקים לכל 5 מחלקות V2. מיפוי השמות (`get_length→get_width`) חוזר ב-V2Slot, V2Cutout, V2Irregular.

### 3. isinstance במערכת המומחה
ה-`isinstance` ב-`expert_system_process` הוא סימן שהפולימורפיזם **לא מלא**. Feature מגדיר רק `get_x()`, `get_y()` — אבל כל סוג מוסיף מתודות אחרות. אין ממשק אחיד.

### 4. צימוד הדוק (Tight Coupling)
V1Slot יודע בדיוק אילו פונקציות V1 לקרוא. אם V1 ישתנה — צריך לשנות את כל 5 מחלקות V1xxx.  
V2Slot יודע בדיוק את הממשק של OOGFeature. אם V2 ישתנה — צריך לשנות את כל 5 מחלקות V2xxx.

### 5. שני ממדים של שונות שאינם מופרדים
הבעיה הזו יש לה **שני צירי שינוי**:
- **מה** (סוג ה-feature: slot, hole, cutout...)
- **מאיפה** (גרסת ה-CAD/CAM: V1, V2, V3...)

פרק 4 מטפל בשניהם **באותו עץ ירושה**. זה עובד אבל מייצר N×M מחלקות.

---

## למה פרק 12 צריך ארכיטקטורה טובה יותר

פרק 12 ישתמש ב**דפוס Bridge** כדי **להפריד את שני הממדים**:
- ציר אחד: Feature → Slot, Hole, Cutout... (מה שמערכת המומחה צריכה)
- ציר שני: Implementation → V1Imp, V2Imp, V3Imp... (איך מחלצים נתונים)

בנוסף:
- **Facade** יעטוף את V1 הפרוצדורלי בממשק OO אחיד
- **Adapter** יתאים את ממשק V2 OOG לממשק שלנו

התוצאה: במקום `6 + 5N` מחלקות, נקבל מבנה שגדל **לינארית** עם כל גרסה חדשה — **לא מכפלתית**.